In [1]:
import fitz
import requests
import io
import pickle
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import re
from bs4 import BeautifulSoup
import pandas as pd
import nltk
#!pip install sddk
import sddk # our package, if missing, uncomment the previous line
import requests
import getpass
from bs4 import BeautifulSoup
from requests_oauthlib import OAuth1
import zipfile
import io
import os

In [5]:
# accessing owncloud.cesnet.cz with sddk package
user = input("Insert your Username code (a long string of characters and numbers): ")
password = getpass.getpass("Insert your Password: ")
s = requests.Session() # create session
s.auth = (user, password)

In [6]:
resp = s.get("https://owncloud.cesnet.cz/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317181.zip")
resp

<Response [200]>

In [7]:
with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        # List the contents of the ZIP file
        print("Files in the ZIP archive:")
        print(z.namelist())

        # Extract contents to a folder (e.g., "./extracted_files")
        extract_dir = "../data/test_pdf+xml/"
        print(f"Extracting files to: {extract_dir}")
        z.extractall(extract_dir)
        print("Extraction complete!")


Files in the ZIP archive:
['/log.txt', '1698730/Dorn1569_Artificii_chymistici_MDZ_MBS.pdf', '1698730/Dorn1569_Artificii_chymistici_MDZ_MBS/metadata.xml']
Extracting files to: ../data/test_pdf+xml/
Extraction complete!


In [8]:
base_url = "https://owncloud.cesnet.cz/"
resp = s.request("PROPFIND", base_url + "/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/")
resp

<Response [207]>

In [9]:
soup = BeautifulSoup(resp.text, "xml")
all_items = soup.find_all("d:response")

In [10]:
all_items

[<d:response><d:href>/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/</d:href><d:propstat><d:prop><d:getlastmodified>Tue, 01 Apr 2025 08:31:17 GMT</d:getlastmodified><d:resourcetype><d:collection/></d:resourcetype><d:quota-used-bytes>18869400398</d:quota-used-bytes><d:quota-available-bytes>85892884347</d:quota-available-bytes><d:getetag>"67eba4550875f"</d:getetag></d:prop><d:status>HTTP/1.1 200 OK</d:status></d:propstat></d:response>,
 <d:response><d:href>/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15316988.zip</d:href><d:propstat><d:prop><d:getlastmodified>Mon, 03 Mar 2025 08:32:00 GMT</d:getlastmodified><d:getcontentlength>202309938</d:getcontentlength><d:resourcetype/><d:getetag>"3b548d4f5ff6de98e4d43492e9858a75"</d:getetag><d:getcontenttype>application/zip</d:getcontenttype></d:prop><d:status>HTTP/1.1 200 OK</d:status></d:propstat></d:response>,
 <d:response><d:href>/remote.php/dav/

In [11]:
hrefs = []
for item in all_items:
     href = item.find("d:href").text
     hrefs.append(href)
hrefs

['/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15316988.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317065.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317181.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317359.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317689.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15318056.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15318280.zip',
 '/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared

In [12]:
target_dir = "/srv/data/tome/tome-corpus/emlap_raw_2025-04-03"
try:
    os.mkdir(target_dir)
except FileExistsError:
    print("Directory already exists.")

In [13]:
hrefs[1]

'/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15316988.zip'

In [14]:
resp = s.get(base_url + href)

In [15]:
base_url = "https://owncloud.cesnet.cz/"
for href in hrefs:
    if ".zip" in href:
        try:
            resp = s.get(base_url + href)
            with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
                # Extract contents to a folder (e.g., "./extracted_files")
                extract_dir = "../data/large_files/emlap_raw_2025-03-10/"
                z.extractall(extract_dir)
                print(href + " - Extraction complete!")
        except:
            print(href + " - Extraction failed!")

/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15316988.zip - Extraction complete!
/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317065.zip - Extraction complete!
/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317181.zip - Extraction complete!
/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317359.zip - Extraction complete!
/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15317689.zip - Extraction complete!
/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15318056.zip - Extraction complete!
/remote.php/dav/files/1fcd50da27c3473f1479718327a43e1a5426eac1/Shared/TOME/EMLAP%20raw/PDF/export_job_15318280.zip - Extraction complete!
/remote.php/dav/files/1fcd50da27c3

In [17]:
len(os.listdir("../data/large_files/emlap_raw_2025-03-10/"))

77

In [18]:
source_dir = "/srv/data/tome/tome-corpus/emlap_raw_2025-03-10/"
for dir in os.listdir(source_dir):
    try:
        print([f for f in os.listdir(source_dir + dir) if ".pdf" in f][0])
    except:
        pass

DuChesne1575_Ad_Iacobi_Auberti_MDZ_Augsburg.pdf
Vadis1595_Dialogus_IA_Wellcome.pdf
Pantheus1530_Voarchadumia_ONB.pdf
Ventura1571_De_ratione_conficiendi_lapis_MBZ_MBS.pdf
Paracelsus1560_Libri_quatuor_de_vita_longa_MDZ_MBS.pdf
Mirandola1586_De_auro_libri_tres_MDZ_MBS.pdf
Auriferae_artisI1572_MBZ_Augsburg.pdf
Fanianus1560_De_arte_metallicae_ONB_pdf.pdf
Rupescissa1561_De_Consideratione_Quintae_essentie_rerum_GB.pdf
Senior1560_De_chemia_senioris_MDZ_MBS.pdf
Gessner1569_Thesaurus_Euonymi_Philiatri_Liber_Secundus_MDZ_MBS.pdf
Anon1550_De_alchemia_opuscula_MDZ_MBS.pdf
Gessner1552_Thesaurus_Euonymi_Philiatri_ER_ZZ.pdf
Dorn1578_Theophrasti_Germani_Principis_MDZ_MBS.pdf
Portaleone1584_De_auro_dialogi_tres_ONB.pdf
De_alchemia1541_MDZ_MBS.pdf
Claveus1598_Apologia_crysopoeiae_MDZ_MBS.pdf
Pseudo-Paracelsus1568_Pyrophilia_vexationumque_ONB.pdf
Paracelsus1553_Labyrinthus_medicorum_errantium_MDZ_MBS_pdf.pdf
Pantheus1519_Commentarium_Transmutationis_Metallicae_MDZ.pdf
DuChesne1575_Sclopetarius_MDZ_Augsbur